# 🏡 San Antonio Automated Apartment & Townhome Hunter (v4 — Persistent Taste Memory Edition)
### What's Fixed & Upgraded in v4:
1. **Fixed Missing Step 4 Bug:** In v3, the scraper execution cell (`Step 4`) was accidentally overwritten by `Step 5`, which caused `NameError: name 'scored_df' is not defined`. All 7 steps are now cleanly separated and verified.
2. **Persistent Taste Memory (`preferences.json`):** Stores your **Rejected Addresses**, **Liked Addresses**, **Disliked Kitchen Photos**, **Liked Kitchen Photos**, and **Dealbreaker Keywords** on Google Drive / GitHub so the tool gets smarter every time you run it.
3. **Interactive Visual Audit + 1-Line Feedback Loop (`Step 6`):** Open `outputs/visual_kitchen_audit.html` to see each candidate's exact kitchen photo and 5-axis sub-scores (`Appliances`, `Counters`, `Cabinets`, `Layout`, `Flooring`). If you disagree with any score, log it in `Step 6` and CLIP will use that photo as a negative/positive vector anchor on all future runs!

In [1]:
# @title Step 0: Install Required Packages
!pip install -q homeharvest osmnx geopandas folium shapely scipy requests pyyaml transformers torch pillow scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.5/149.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 78.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


In [2]:
# @title Step 1: Configuration & Persistent Memory Loader (`preferences.json`)
import os
import re
import json
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely.geometry import Point
from scipy.spatial import cKDTree
from homeharvest import scrape_property

# Optional: Set USE_GOOGLE_DRIVE_MEMORY = True in Colab to persist preferences across Colab resets!
USE_GOOGLE_DRIVE_MEMORY = True
PREF_FILE = 'data/preferences.json'

if USE_GOOGLE_DRIVE_MEMORY:
    from google.colab import drive
    drive.mount('/content/drive')
    PREF_FILE = '/content/drive/MyDrive/LoveWhereYouLive_preferences.json'

CONFIG = {
    'workplace': {'name': 'Southwest Research Institute (SwRI)', 'coords': (29.4485, -98.6105)},
    'costco_locations': [
        {'name': 'Costco NW (5611 UTSA Blvd)', 'coords': (29.5762, -98.6001)},
        {'name': 'Costco West (191 W Loop 1604 S)', 'coords': (29.4316, -98.7115)},
        {'name': 'Costco North Central (1201 N Loop 1604 E)', 'coords': (29.6095, -98.4716)},
        {'name': 'Costco NE Selma (15330 IH-35 N)', 'coords': (29.5794, -98.3188)}
    ],
    'filters': {'location': 'San Antonio, TX', 'min_rent': 1200, 'max_rent': 2200, 'min_beds': 2, 'min_baths': 1.5, 'past_days': 30},
    'cutoffs': {'require_in_unit_laundry_and_dishwasher': True, 'strict_appliance_text_match': False, 'max_swri_commute_mins': 28.0, 'max_grocery_dist_m': 2400.0, 'max_park_trail_dist_m': 1600.0},
    'weights': {
        'commute_score': 0.16,
        'grocery_score': 0.11,
        'park_trail_score': 0.11,
        'pollution_highway_score': 0.07,
        'modern_interior_score': 0.10,
        'garage_bike_score': 0.07,
        'costco_score': 0.07,
        'water_softener_score': 0.05,
        'pet_flooring_score': 0.05,
        'effective_rent_score': 0.05,
        'noise_buffer_score': 0.05,
        'recency_score': 0.06
    }
}

import os
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

PREF_PATH = "data/preferences.json"

def load_preferences(path=PREF_PATH):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        default_prefs = {
            "version": "4.0",
            "taste_profile": {
                "liked_kitchen_photo_urls": [],
                "disliked_kitchen_photo_urls": [],
                "liked_property_addresses": [],
                "rejected_property_addresses": [],
                "rejection_reasons_log": [],
                "excluded_grocery_chains": ["walmart", "dollar general", "family dollar"],
                "dealbreaker_keywords": ["shared laundry", "coin-operated", "no washer", "student housing", "room for rent"]
            },
            "learned_weights_override": {}
        }
        with open(path, "w") as f:
            json.dump(default_prefs, f, indent=2)
        return default_prefs
    with open(path, "r") as f:
        return json.load(f)

def save_preferences(prefs, path=PREF_PATH):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(prefs, f, indent=2)

def record_feedback(
    address=None,
    liked=False,
    kitchen_photo_url=None,
    reason="",
    new_dealbreaker_keyword=None,
    path=PREF_PATH
):
    """Records thumbs-up / thumbs-down feedback on a property or kitchen photo into persistent JSON memory."""
    prefs = load_preferences(path)
    tp = prefs["taste_profile"]

    if address:
        if liked and address not in tp["liked_property_addresses"]:
            tp["liked_property_addresses"].append(address)
        elif not liked and address not in tp["rejected_property_addresses"]:
            tp["rejected_property_addresses"].append(address)

    if kitchen_photo_url:
        if liked and kitchen_photo_url not in tp["liked_kitchen_photo_urls"]:
            tp["liked_kitchen_photo_urls"].append(kitchen_photo_url)
        elif not liked and kitchen_photo_url not in tp["disliked_kitchen_photo_urls"]:
            tp["disliked_kitchen_photo_urls"].append(kitchen_photo_url)

    if new_dealbreaker_keyword:
        kw = new_dealbreaker_keyword.lower().strip()
        if kw and kw not in tp["dealbreaker_keywords"]:
            tp["dealbreaker_keywords"].append(kw)

    if reason:
        tp["rejection_reasons_log"].append({
            "address": address,
            "liked": liked,
            "kitchen_photo_url": kitchen_photo_url,
            "reason": reason
        })

    save_preferences(prefs, path)
    print(f"Memory updated! Total Liked Properties: {len(tp['liked_property_addresses'])}, Rejected: {len(tp['rejected_property_addresses'])}")
    print(f"Kitchen Anchors -> Liked: {len(tp['liked_kitchen_photo_urls'])}, Disliked: {len(tp['disliked_kitchen_photo_urls'])}")
    return prefs

def calibrate_weights_from_ratings(scored_df, user_ratings_dict, base_weights, path=PREF_PATH):
    """
    Active Learning Weight Tuner:
    Given a dictionary of {address_substring: user_score_0_to_100}, fits a Ridge regression
    over the sub-scores to learn which criteria actually drive your personal satisfaction,
    and saves the updated weights into `data/preferences.json`.
    """
    score_cols = list(base_weights.keys())
    X, y = [], []
    for addr_sub, target_score in user_ratings_dict.items():
        matches = scored_df[scored_df['formatted_address'].astype(str).str.contains(addr_sub, case=False, na=False)]
        if len(matches) > 0:
            row = matches.iloc[0]
            X.append([float(row.get(c, 50.0)) for c in score_cols])
            y.append(float(target_score))

    if len(X) < 3:
        print("Need at least 3 rated properties to auto-tune weights! Keeping current weights.")
        return base_weights

    model = Ridge(alpha=10.0, positive=True)
    model.fit(np.array(X), np.array(y))
    coefs = model.coef_
    if coefs.sum() > 0:
        learned = coefs / coefs.sum()
        # Blend 50% prior config weights + 50% learned empirical weights for stability
        new_weights = {}
        for col, lw in zip(score_cols, learned):
            new_weights[col] = round(float(0.50 * base_weights[col] + 0.50 * lw), 3)
        # Normalize to sum to 1.0
        total = sum(new_weights.values())
        new_weights = {k: round(v / total, 3) for k, v in new_weights.items()}
        prefs = load_preferences(path)
        prefs["learned_weights_override"] = new_weights
        save_preferences(prefs, path)
        print("Learned new weights from your ratings:", new_weights)
        return new_weights
    return base_weights

PREFS = load_preferences(PREF_FILE)
if PREFS.get('learned_weights_override'):
    CONFIG['weights'].update(PREFS['learned_weights_override'])
print(f'Loaded Configuration & Persistent Memory from {PREF_FILE}!')
print(f"Memory Stats -> Liked Kitchens: {len(PREFS['taste_profile']['liked_kitchen_photo_urls'])}, Disliked Kitchens: {len(PREFS['taste_profile']['disliked_kitchen_photo_urls'])}, Rejected Addresses: {len(PREFS['taste_profile']['rejected_property_addresses'])}")

Mounted at /content/drive
Loaded Configuration & Persistent Memory from /content/drive/MyDrive/LoveWhereYouLive_preferences.json!
Memory Stats -> Liked Kitchens: 0, Disliked Kitchens: 0, Rejected Addresses: 0


In [3]:
# @title Step 2: Build or Load Cached Spatial KD-Trees (Groceries [No Walmart], Parks, Highways, Railways)
import os
import pickle
import numpy as np
import geopandas as gpd
import osmnx as ox
from scipy.spatial import cKDTree

loc = CONFIG['filters']['location']
CACHE_DIR = "cache"
if USE_GOOGLE_DRIVE_MEMORY:
    CACHE_DIR = "/content/drive/MyDrive/LoveWhereYouLive_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CACHE_FILE = os.path.join(CACHE_DIR, f"osm_trees_{loc.replace(', ', '_').replace(' ', '_').lower()}.pkl")
FORCE_REFRESH_CACHE = False  # Set to True only if you want to force re-downloading fresh OSM data

def get_osm_tree(place, tags):
    gdf = ox.features_from_place(place, tags=tags).to_crs(epsg=32614)
    xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in gdf.geometry])
    names = gdf['name'].fillna('Unnamed').tolist() if 'name' in gdf.columns else ['Unnamed'] * len(xy)
    return cKDTree(xy), names

if os.path.exists(CACHE_FILE) and not FORCE_REFRESH_CACHE:
    print(f"⚡ Loading cached spatial KD-Trees from {CACHE_FILE} (instant)...")
    with open(CACHE_FILE, "rb") as f:
        TREES = pickle.load(f)
    print(f"Loaded {len(TREES)} spatial layers from cache in <0.1s!")
else:
    print(f"Fetching OpenStreetMap geospatial layers for {loc} (this only runs once, then caches)...")
    TREES = {
        'grocery': get_osm_tree(loc, {'shop': ['supermarket', 'grocery', 'greengrocer']}),
        'park': get_osm_tree(loc, {'leisure': ['park', 'nature_reserve'], 'highway': ['cycleway', 'path', 'footway']}),
        'highway': get_osm_tree(loc, {'highway': ['motorway', 'trunk']}),
        'railway': get_osm_tree(loc, {'railway': 'rail'})
    }
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(TREES, f)
    print(f"Saved spatial KD-Trees to persistent cache: {CACHE_FILE}")


⚡ Loading cached spatial KD-Trees from /content/drive/MyDrive/LoveWhereYouLive_cache/osm_trees_san_antonio_tx.pkl (instant)...
Loaded 4 spatial layers from cache in <0.1s!


In [4]:
# @title Step 3: Register All 12 Modular Criteria Functions
import math
import re
import numpy as np
import pandas as pd

CIRCUITY_FACTOR = 1.32
WALK_SPEED_M_MIN = 80.0
BIKE_SPEED_M_MIN = 250.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def criterion_required_appliances_gatekeeper(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    strict = ctx['config']['cutoffs'].get('strict_appliance_text_match', False)

    shared_only_flags = ['shared laundry', 'laundromat on site', 'no washer', 'no laundry in unit', 'coin-operated laundry']
    if any(flag in desc for flag in shared_only_flags):
        return {'appliances_status': 'REJECT: Shared/No In-Unit Laundry', 'passes_appliance_gate': False}

    in_unit_terms = ['in-unit', 'in unit', 'washer/dryer', 'washer and dryer', 'w/d included', 'w/d connections', 'laundry room', 'utility room', 'full-size washer']
    has_in_unit = any(t in desc for t in in_unit_terms)
    has_dishwasher = 'dishwasher' in desc or ('stainless' in desc and 'appliances' in desc) or (pd.notna(year_built) and year_built >= 2005)

    if has_in_unit and has_dishwasher:
        return {'appliances_status': 'Verified (In-Unit W/D + Dishwasher)', 'passes_appliance_gate': True}
    elif strict:
        return {'appliances_status': 'REJECT: Unconfirmed in Text', 'passes_appliance_gate': False}
    return {'appliances_status': 'Likely (Verify W/D in Photos)', 'passes_appliance_gate': True}

def criterion_garage_and_bike_storage(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    prop_type = str(row.get('style', '') or row.get('type', '') or '').lower()
    parking_spaces = row.get('parking_garage', np.nan)

    score = 30.0
    features = []
    if pd.notna(parking_spaces) and float(parking_spaces) >= 1:
        score += 45.0
        features.append(f'{int(float(parking_spaces))}-Car Garage')
    elif any(k in desc for k in ['attached garage', 'private garage', '1-car garage', '2-car garage', 'two-car garage', 'single car garage']):
        score += 45.0
        features.append('Private/Attached Garage')
    elif 'carport' in desc or 'covered parking' in desc:
        score += 20.0
        features.append('Covered/Carport')

    if any(k in desc for k in ['bike storage', 'bicycle storage', 'bike rack', 'bike repair', 'storage unit', 'storage closet']):
        score += 25.0
        features.append('Dedicated Bike/Gear Storage')
    if 'townhome' in prop_type or 'townhouse' in prop_type or 'townhome' in desc or 'first floor' in desc:
        score += 15.0
        features.append('Townhome/Ground Access')

    return {
        'garage_bike_features': ', '.join(features) if features else 'Standard Surface Parking',
        'garage_bike_score': round(float(np.clip(score, 0.0, 100.0)), 1)
    }

def criterion_costco_proximity(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    best_name, min_road_km = '', 999.0
    for c in ctx['config']['costco_locations']:
        clat, clon = c['coords']
        road_km = haversine_km(lat, lon, clat, clon) * 1.28
        if road_km < min_road_km:
            min_road_km, best_name = road_km, c['name']
    est_drive_mins = (min_road_km / 48.0) * 60.0 + 2.0
    score = np.clip(100 - ((est_drive_mins - 7.0) / (25.0 - 7.0)) * 100, 0, 100)
    return {
        'nearest_costco': best_name,
        'costco_drive_mins': round(est_drive_mins, 1),
        'costco_score': round(float(score), 1)
    }

def criterion_grocery_proximity(row, ctx):
    tree, names = ctx['trees']['grocery']
    distances, indices = tree.query([row['utm_x'], row['utm_y']], k=min(5, len(names)))

    if not isinstance(indices, np.ndarray):
        indices = [indices]
        distances = [distances]

    best_idx = indices[0]
    best_dist = distances[0]
    best_name = names[best_idx]

    for idx, dist in zip(indices, distances):
        name = names[idx].lower()
        if 'walmart' not in name:
            best_idx = idx
            best_dist = dist
            best_name = names[idx]
            break

    network_dist_m = best_dist * CIRCUITY_FACTOR
    walk_mins = network_dist_m / WALK_SPEED_M_MIN
    bike_mins = network_dist_m / BIKE_SPEED_M_MIN
    score = np.clip(100 - ((network_dist_m - 400) / (2400 - 400)) * 100, 0, 100)
    return {
        'nearest_grocery': best_name,
        'grocery_dist_m': round(network_dist_m),
        'grocery_walk_mins': round(walk_mins, 1),
        'grocery_bike_mins': round(bike_mins, 1),
        'grocery_score': round(float(score), 1)
    }

def criterion_park_trail_proximity(row, ctx):
    tree, names = ctx['trees']['park']
    dist_euclidean, idx = tree.query([row['utm_x'], row['utm_y']])
    network_dist_m = dist_euclidean * 1.25
    walk_mins = network_dist_m / WALK_SPEED_M_MIN
    score = np.clip(100 - ((network_dist_m - 250) / (1600 - 250)) * 100, 0, 100)
    return {
        'nearest_park_trail': names[idx],
        'park_dist_m': round(network_dist_m),
        'park_walk_mins': round(walk_mins, 1),
        'park_trail_score': round(float(score), 1)
    }

def criterion_swri_rush_hour_commute(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    swri_lat, swri_lon = ctx['config']['workplace']['coords']
    road_km = haversine_km(lat, lon, swri_lat, swri_lon) * 1.30
    if lon < -98.68:
        avg_speed_kmh, corridor_note = 28.0, 'West Inbound (Heavy Culebra/1604 Traffic)'
    elif lon > -98.58:
        avg_speed_kmh, corridor_note = 50.0, 'Reverse Commute (Central/North)'
    else:
        avg_speed_kmh, corridor_note = 40.0, 'Inner West / Loop 410 Local'
    est_commute_mins = (road_km / avg_speed_kmh) * 60.0 + 3.0
    score = np.clip(100 - ((est_commute_mins - 8.0) / (30.0 - 8.0)) * 100, 0, 100)
    return {
        'swri_road_km': round(road_km, 1),
        'swri_rush_hour_mins': round(est_commute_mins, 1),
        'commute_corridor': corridor_note,
        'commute_score': round(float(score), 1)
    }

def criterion_pollution_and_highway_buffer(row, ctx):
    tree, _ = ctx['trees']['highway']
    hwy_dist_m, _ = tree.query([row['utm_x'], row['utm_y']])
    if hwy_dist_m < 100:
        score = 15.0
    elif hwy_dist_m < 250:
        score = 50.0 + ((hwy_dist_m - 100) / 150.0) * 35.0
    else:
        score = np.clip(85.0 + ((hwy_dist_m - 250) / 500.0) * 15.0, 85.0, 100.0)
    return {
        'highway_buffer_m': round(hwy_dist_m),
        'pollution_highway_score': round(float(score), 1)
    }

def criterion_modern_appliances_and_open_floorplan(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    score, matched_tags = 50.0, []
    positive_keywords = {
        'open floor': 18, 'open-concept': 18, 'open concept': 18,
        'stainless steel': 15, 'stainless appliances': 15,
        'quartz': 12, 'granite': 10, 'kitchen island': 12, 'island kitchen': 12,
        'gas range': 12, 'gas stove': 12, 'modern': 8, 'renovated': 10
    }
    for kw, pts in positive_keywords.items():
        if kw in desc:
            score += pts
            matched_tags.append(kw)
    if pd.notna(year_built) and year_built >= 2018:
        score += 20
        matched_tags.append(f'built_{int(year_built)}')
    return {
        'interior_tags': ', '.join(matched_tags) if matched_tags else 'Standard/Unlisted',
        'modern_interior_score': round(float(np.clip(score, 10.0, 100.0)), 1)
    }

def criterion_water_softener_and_filtration(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score, tags = 40.0, []
    softener_keywords = {
        'water softener': 40, 'softener loop': 35, 'softened water': 35,
        'reverse osmosis': 25, 'ro system': 25, 'filtration system': 20,
        'water filtration': 20, 'tankless water heater': 15
    }
    for kw, pts in softener_keywords.items():
        if kw in desc:
            score += pts
            tags.append(kw)
    return {
        'water_softener_tags': ', '.join(tags) if tags else 'Standard Hard Water (No Softener Listed)',
        'water_softener_score': round(float(np.clip(score, 10.0, 100.0)), 1)
    }

def criterion_pet_friendly_and_hard_flooring(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score, tags = 60.0, []
    hard_flooring_terms = ['vinyl plank', 'lvp', 'wood floor', 'hardwood', 'tile flooring', 'stained concrete', 'no carpet']
    carpet_flags = ['wall-to-wall carpet', 'carpet in bedrooms', 'new carpet']
    cat_terms = ['cats allowed', 'cat friendly', 'pets allowed', 'pet friendly', 'no pet rent']

    if any(term in desc for term in hard_flooring_terms):
        score += 25.0
        tags.append('Hard Flooring / LVP')
    elif any(flag in desc for flag in carpet_flags):
        score -= 20.0
        tags.append('Carpeted Bedrooms')
    if any(term in desc for term in cat_terms):
        score += 15.0
        tags.append('Pet/Cat Friendly')

    return {
        'pet_flooring_tags': ', '.join(tags) if tags else 'Standard Flooring',
        'pet_flooring_score': round(float(np.clip(score, 10.0, 100.0)), 1)
    }

def criterion_advanced_junk_fee_parser(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    fee_patterns = r'\$(\d+)\s*(?:/mo)?\s*(?:valet|trash|waste|package|internet|tech|pest|amenity|fee)'
    matches = re.findall(fee_patterns, desc)
    extracted_fees = sum([int(m) for m in matches])
    utilities_included = any(k in desc for k in ['water included', 'trash included', 'internet included', 'free wifi', 'all bills paid'])

    score = 100.0 - min(extracted_fees, 50.0)
    if utilities_included:
        score += 10.0
    return {
        'extracted_monthly_fees': extracted_fees,
        'utilities_included': utilities_included,
        'effective_rent_score': round(float(np.clip(score, 30.0, 100.0)), 1)
    }

def criterion_spatial_noise_buffer(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    utm_x, utm_y = row['utm_x'], row['utm_y']

    rail_dist_m = 9999.0
    if 'railway' in ctx['trees']:
        rail_tree, _ = ctx['trees']['railway']
        rail_dist_m, _ = rail_tree.query([utm_x, utm_y])

    sat_coords = (29.5338, -98.4698)
    sat_dist_km = haversine_km(lat, lon, sat_coords[0], sat_coords[1])

    score, notes = 90.0, []
    if rail_dist_m < 200:
        score -= 40.0
        notes.append(f'Close to Railway ({round(rail_dist_m)}m)')
    elif rail_dist_m < 500:
        score -= 15.0
        notes.append(f'Moderate Rail Proximity ({round(rail_dist_m)}m)')
    if sat_dist_km < 6.0:
        score -= 25.0
        notes.append(f'SAT Airport Approach ({round(sat_dist_km, 1)}km)')

    return {
        'railway_dist_m': round(rail_dist_m) if rail_dist_m < 9000 else 9999,
        'sat_airport_dist_km': round(sat_dist_km, 1),
        'noise_buffer_notes': ', '.join(notes) if notes else 'Low Noise Corridor',
        'noise_buffer_score': round(float(np.clip(score, 20.0, 100.0)), 1)
    }


def criterion_year_built_recency(row, ctx):
    """CRITERION: Favors newer construction properties (year_built) to ensure modern structural standards."""
    year_built = row.get('year_built', np.nan)
    if pd.isna(year_built):
        return {'property_age_years': 'Unknown', 'recency_score': 50.0}
    current_year = 2026
    age_years = max(0, current_year - int(year_built))
    if age_years <= 3:
        score = 100.0
    elif age_years <= 7:
        score = 85.0
    elif age_years <= 12:
        score = 70.0
    elif age_years <= 20:
        score = 50.0
    else:
        score = np.clip(100.0 - (age_years * 2.5), 15.0, 45.0)
    return {'property_age_years': age_years, 'recency_score': round(float(score), 1)}

ACTIVE_CRITERIA = [
    criterion_year_built_recency,
    criterion_required_appliances_gatekeeper,
    criterion_garage_and_bike_storage,
    criterion_costco_proximity,
    criterion_grocery_proximity,
    criterion_park_trail_proximity,
    criterion_swri_rush_hour_commute,
    criterion_pollution_and_highway_buffer,
    criterion_modern_appliances_and_open_floorplan,
    criterion_water_softener_and_filtration,
    criterion_pet_friendly_and_hard_flooring,
    criterion_advanced_junk_fee_parser,
    criterion_spatial_noise_buffer
]

print(f'Registered {len(ACTIVE_CRITERIA)} criteria functions!')

Registered 13 criteria functions!


In [5]:
# @title Step 4: Scrape Listings & Compute Spatial/Text Scores (`scored_df`)
print('Scraping active rental listings in San Antonio...')
raw = scrape_property(location=CONFIG['filters']['location'], listing_type='for_rent', past_days=CONFIG['filters']['past_days'])
df = raw.dropna(subset=['latitude', 'longitude', 'list_price']).copy()

f = CONFIG['filters']
df = df[
    (df['list_price'] >= f['min_rent']) &
    (df['list_price'] <= f['max_rent']) &
    (df['beds'].fillna(0) >= f['min_beds']) &
    (df['full_baths'].fillna(0) >= 1)
].copy()

# Filter out any addresses or dealbreaker keywords stored in your Persistent Memory!
PREFS = load_preferences(PREF_FILE)
rejected_addrs = set(PREFS['taste_profile'].get('rejected_property_addresses', []))
dealbreakers = [kw.lower() for kw in PREFS['taste_profile'].get('dealbreaker_keywords', [])]

if rejected_addrs:
    before_len = len(df)
    df = df[~df['formatted_address'].isin(rejected_addrs)].copy()
    print(f'Memory Filter: Dropped {before_len - len(df)} previously rejected properties.')

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326').to_crs(epsg=32614)
df['utm_x'], df['utm_y'] = gdf.geometry.x, gdf.geometry.y

ctx = {'config': CONFIG, 'trees': TREES}
results = []
weights = CONFIG['weights']

for _, row in df.iterrows():
    desc_lower = str(row.get('description', '') or '').lower()
    if any(db in desc_lower for db in dealbreakers):
        results.append({'passes_appliance_gate': False})
        continue
    m = {}
    for fn in ACTIVE_CRITERIA:
        m.update(fn(row, ctx))
    if m.get('passes_appliance_gate', True):
        m['composite_score'] = round(sum(m.get(k, 50.0) * w for k, w in weights.items()), 1)
    results.append(m)

full_scored = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
c_cfg = CONFIG['cutoffs']
scored_df = full_scored[
    (full_scored['passes_appliance_gate'] == True) &
    (full_scored['swri_rush_hour_mins'] <= c_cfg['max_swri_commute_mins']) &
    (full_scored['grocery_dist_m'] <= c_cfg['max_grocery_dist_m']) &
    (full_scored['park_dist_m'] <= c_cfg['max_park_trail_dist_m'])
].sort_values(by='composite_score', ascending=False).reset_index(drop=True)

print(f'Step 4 Complete! `scored_df` created with {len(scored_df)} surviving candidates.')
scored_df[['composite_score', 'formatted_address', 'list_price', 'property_age_years', 'swri_rush_hour_mins', 'nearest_grocery', 'nearest_costco']].head(10)

Scraping active rental listings in San Antonio...


/tmp/ipykernel_1604/3293029402.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['beds'].fillna(0) >= f['min_beds']) &
/tmp/ipykernel_1604/3293029402.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['full_baths'].fillna(0) >= 1)


Step 4 Complete! `scored_df` created with 202 surviving candidates.


,composite_score,formatted_address,list_price,property_age_years,swri_rush_hour_mins,nearest_grocery,nearest_costco
0,76.1,"5142 Kelwood St Unit 102, San Antonio, TX, 78250",1600,1,15.1,H-E-B,Costco West (191 W Loop 1604 S)
1,71.2,"1022 Avocet, San Antonio, TX, 78245",1700,21,13.1,H-E-B Plus!,Costco West (191 W Loop 1604 S)
2,70.9,"9627 Summer Vail, San Antonio, TX, 78251",1595,23,15.7,Sprouts Farmers Market,Costco West (191 W Loop 1604 S)
3,70.6,"11819 Zarkava Way, San Antonio, TX, 78230",2195,2,26.1,Trader Joe's,Costco NW (5611 UTSA Blvd)
4,70.5,"7731 Bohemian Grv, San Antonio, TX, 78227",1800,2,10.1,H-E-B Plus!,Costco West (191 W Loop 1604 S)
5,69.9,"4121 Copano Bay, San Antonio, TX, 78229",1700,21,17.8,NOOR GROCERY,Costco NW (5611 UTSA Blvd)
6,69.8,"7610 Bohemian Grv, San Antonio, TX, 78227",1495,1,9.7,H-E-B Plus!,Costco West (191 W Loop 1604 S)
7,69.3,"5015 Shara Mist, San Antonio, TX, 78251",1495,Unknown,16.3,H-E-B,Costco West (191 W Loop 1604 S)
8,68.7,"4010 High Ridge Cir, San Antonio, TX, 78229",1950,48,14.8,Minnano Japanese Grocery,Costco NW (5611 UTSA Blvd)
9,68.5,"2031 Silver Oaks, San Antonio, TX, 78213",1900,3,24.0,La Michoacana Meat Market,Costco NW (5611 UTSA Blvd)


In [6]:
# @title Step 5: Run 5-Axis Kitchen Vision Grader (Uses Your Liked/Disliked Memory!) + Export Audit HTML
import io
import os
import requests
from PIL import Image
import numpy as np
import pandas as pd

def score_photos_with_local_clip(
    df_top,
    max_candidates=25,
    liked_image_urls=None,
    disliked_image_urls=None,
    export_audit_html="outputs/visual_kitchen_audit.html"
):
    """
    Robust 3-Stage Local Vision Grader (OpenAI CLIP):
    1. Strict Room Isolation: Separates actual Kitchen photos from Living Room/Flooring photos
       (so a sunlit living room can NEVER mask an outdated kitchen via `max()`).
    2. 5-Axis Pairwise Feature Decomposition (Binary Softmax per physical attribute):
       - Axis A (Appliances): Stainless steel vs. White/black plastic or exposed electric coils
       - Axis B (Countertops): Thick quartz/granite stone slab vs. Thin laminate/formica/tile
       - Axis C (Cabinetry): Modern flat-panel/shaker cabinets vs. Dated 1990s oak/honey-wood cabinets
       - Axis D (Layout): Open-concept island/bar layout vs. Cramped closed galley wall kitchen
       - Axis E (Flooring): Hardwood / Luxury Vinyl Plank (LVP) vs. Wall-to-wall carpet
    3. Optional Few-Shot Personal Calibration:
       - If you pass 1-3 URLs of kitchens you LOVE (`liked_image_urls`) and 1-3 kitchens you HATE
         (`disliked_image_urls`), it computes exact cosine similarity in CLIP embedding space
         to your personal taste!
    4. Transparency Audit HTML:
       - Outputs `outputs/visual_kitchen_audit.html` showing the exact photo selected as the
         kitchen and every sub-score so you can immediately see why a unit scored what it did.
    """
    try:
        from transformers import CLIPProcessor, CLIPModel
        import torch
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "-q", "transformers", "torch", "pillow"])
        from transformers import CLIPProcessor, CLIPModel
        import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Robust Multi-Axis CLIP Vision Engine on {device}...")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    # Stage 1: Strict Room Classification
    room_labels = [
        "an indoor kitchen with stove, refrigerator, sink, countertops, and cabinets", # 0: Kitchen
        "an empty indoor living room or bedroom showing floors and walls",             # 1: Living/Bedroom
        "anindoor bathroom with toilet, shower, or bathtub",                           # 2: Bathroom
        "an outdoor building exterior, swimming pool, patio, street, or grass",        # 3: Exterior
        "a 2D floor plan diagram, map, or text logo"                                   # 4: Diagram
    ]

    # Stage 2: Independent Binary Contrasts for Kitchens (evaluated ONLY on verified kitchen photos)
    kitchen_axes = {
        "appliances": [
            "modern stainless steel kitchen appliances, sleek induction or gas range, stainless refrigerator",
            "old white or black plastic kitchen appliances, exposed electric coil burners, cheap stove"
        ],
        "counters": [
            "luxury thick quartz, marble, or polished granite stone kitchen countertops with undermount sink",
            "cheap laminate formica countertops,dated beige or speckled counter, or old drop-in sink"
        ],
        "cabinets": [
            "modern shaker style or sleek flat-panel contemporary kitchen cabinets with modern hardware",
            "dated 1980s or 1990s orange oak wood cabinets, cheap builder-grade brown wood doors, or worn cabinets"
        ],
        "layout": [
            "spacious open-concept kitchen with a large center island or breakfast bar overlooking the living room",
            "cramped narrow galley kitchen, closed-off small apartment kitchen with no island and tight walls"
        ]
    }

    # Independent Binary Contrast for Flooring (evaluated on kitchen + living room photos)
    flooring_axis = [
        "clean modern luxury vinyl plank flooring, wide-plank hardwood floors, or modern large format tile",
        "wall-to-wall beige carpet, plush bedroom carpeting, or dated linoleum squares"
    ]

    def get_image_embedding(img):
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            feat = model.get_image_features(**inputs)
            feat = feat / feat.norm(p=2, dim=-1, keepdim=True)
        return feat.cpu().numpy()[0]

    def binary_axis_prob(img, pair_prompts):
        inputs = processor(text=pair_prompts, images=img, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits_per_image
            probs = logits.softmax(dim=1).cpu().numpy()[0]
        return float(probs[0]) # Probability of the positive (modern) pole vs negative pole

    # Pre-compute user reference embeddings if provided (Few-Shot Personal Taste Anchors)
    liked_embs, disliked_embs = [], []
    if liked_image_urls:
        for u in liked_image_urls:
            try:
                im = Image.open(io.BytesIO(requests.get(u, timeout=4).content)).convert("RGB")
                liked_embs.append(get_image_embedding(im))
            except Exception:
                pass
    if disliked_image_urls:
        for u in disliked_image_urls:
            try:
                im = Image.open(io.BytesIO(requests.get(u, timeout=4).content)).convert("RGB")
                disliked_embs.append(get_image_embedding(im))
            except Exception:
                pass

    scores = []
    notes = []
    audit_rows = []

    for idx, row in df_top.head(max_candidates).iterrows():
        urls = []
        if pd.notna(row.get('primary_photo')) and str(row.get('primary_photo')).startswith('http'):
            urls.append(str(row['primary_photo']))
        if pd.notna(row.get('alt_photos')):
            alt = [u.strip() for u in str(row['alt_photos']).split(',') if u.strip().startswith('http')]
            urls.extend(alt[:14]) # Scan up to 15 photos so we don't miss the kitchen!

        # Deduplicate while preserving order
        seen = set()
        urls = [u for u in urls if not (u in seen or seen.add(u))][:15]

        if not urls:
            scores.append(45.0)
            notes.append("No photo URLs (Unverified)")
            continue

        kitchen_candidates = [] # (kitchen_prob, url, img)
        living_candidates = []  # (living_prob, url, img)

        for url in urls:
            try:
                resp = requests.get(url, timeout=3.5)
                img = Image.open(io.BytesIO(resp.content)).convert("RGB")
                inputs = processor(text=room_labels, images=img, return_tensors="pt", padding=True).to(device)
                with torch.no_grad():
                    r_probs = model(**inputs).logits_per_image.softmax(dim=1).cpu().numpy()[0]

                k_prob, l_prob, b_prob, ext_prob, diag_prob = [float(x) for x in r_probs]
                # Reject exteriors, bathrooms, and floorplan diagrams
                if ext_prob > 0.35 or diag_prob > 0.35 or b_prob > 0.40:
                    continue
                if k_prob >= 0.38:
                    kitchen_candidates.append((k_prob, url, img))
                elif l_prob >= 0.38:
                    living_candidates.append((l_prob, url, img))
            except Exception:
                continue

        # Sort by highest confidence that the photo is genuinely a kitchen
        kitchen_candidates.sort(key=lambda x: x[0], reverse=True)
        living_candidates.sort(key=lambda x: x[0], reverse=True)

        if not kitchen_candidates:
            # DO NOT reward a listing if it hides or lacks kitchen photos!
            floor_scores = [binary_axis_prob(im, flooring_axis) * 100.0 for _, _, im in living_candidates[:2]]
            fallback_floor = np.mean(floor_scores) if floor_scores else 45.0
            penalized = round(float(np.clip(fallback_floor * 0.65, 25.0, 58.0)), 1)
            scores.append(penalized)
            notes.append(f"No Kitchen Photo Found (Capped: {penalized}/100)")
            continue

        # Evaluate the Top 2 clearest kitchen photos across all 4 kitchen axes
        top_k = kitchen_candidates[:2]
        axis_results = {k: [] for k in kitchen_axes}
        floor_vals = []
        taste_vals = []

        for k_conf, k_url, k_img in top_k:
            for axis_name, pair_prompts in kitchen_axes.items():
                axis_results[axis_name].append(binary_axis_prob(k_img, pair_prompts) * 100.0)
            floor_vals.append(binary_axis_prob(k_img, flooring_axis) * 100.0)

            if liked_embs or disliked_embs:
                emb = get_image_embedding(k_img)
                pos_sim = max([float(np.dot(emb, le)) for le in liked_embs]) if liked_embs else 0.25
                neg_sim = max([float(np.dot(emb, de)) for de in disliked_embs]) if disliked_embs else 0.25
                # Convert cosine margin (-0.15 to +0.15) into 0-100 score
                margin = np.clip((pos_sim - neg_sim) * 350.0 + 50.0, 0.0, 100.0)
                taste_vals.append(float(margin))

        for _, _, l_img in living_candidates[:2]:
            floor_vals.append(binary_axis_prob(l_img, flooring_axis) * 100.0)

        # Aggregate per-axis scores (using mean across kitchen views, never `max()` across non-kitchens)
        app_s = float(np.mean(axis_results["appliances"]))
        cnt_s = float(np.mean(axis_results["counters"]))
        cab_s = float(np.mean(axis_results["cabinets"]))
        lay_s = float(np.mean(axis_results["layout"]))
        flr_s = float(np.mean(floor_vals)) if floor_vals else 50.0

        # Bottleneck / "Weakest Link" Rule:
        # If cabinets, counters, OR appliances are severely dated (< 35/100), cap or heavily penalize!
        raw_weighted = (0.28 * app_s) + (0.26 * cnt_s) + (0.24 * cab_s) + (0.12 * lay_s) + (0.10 * flr_s)
        weakest_kitchen_link = min(app_s, cnt_s, cab_s)
        if weakest_kitchen_link < 35.0:
            # Drag the score down toward the weakest link so old cabinets + new fridge don't get an 85
            raw_weighted = 0.50 * raw_weighted + 0.50 * weakest_kitchen_link

        if taste_vals:
            raw_weighted = 0.70 * raw_weighted + 0.30 * float(np.mean(taste_vals))

        final_v_score = round(float(np.clip(raw_weighted, 10.0, 100.0)), 1)
        scores.append(final_v_score)

        breakdown_str = (
            f"Kitchen {final_v_score}/100 "
            f"[App:{round(app_s)} | Cnt:{round(cnt_s)} | Cab:{round(cab_s)} | Open:{round(lay_s)} | Flr:{round(flr_s)}]"
        )
        notes.append(breakdown_str)

        audit_rows.append({
            "address": row.get("formatted_address", "Listing"),
            "price": row.get("list_price", ""),
            "url": row.get("property_url", "#"),
            "kitchen_img_url": top_k[0][1],
            "kitchen_conf": round(top_k[0][0] * 100),
            "final_score": final_v_score,
            "appliances": round(app_s),
            "counters": round(cnt_s),
            "cabinets": round(cab_s),
            "layout": round(lay_s),
            "flooring": round(flr_s)
        })

    # Export visual audit HTML so the user can inspect the exact kitchen photo & sub-scores
    if export_audit_html and audit_rows:
        os.makedirs(os.path.dirname(export_audit_html), exist_ok=True)
        cards_html = ""
        for r in sorted(audit_rows, key=lambda x: x["final_score"], reverse=True):
            badge_color = "#1b7e3c" if r["final_score"] >= 75 else ("#b57c00" if r["final_score"] >= 55 else "#b82e2e")
            cards_html += f"""
            <div style="border:1px solid #ddd; border-radius:8px; width:340px; overflow:hidden; background:#fff; box-shadow:0 2px 5px rgba(0,0,0,0.08);">
                <img src="{r['kitchen_img_url']}" style="width:100%; height:210px; object-fit:cover;" />
                <div style="padding:12px; font-family:sans-serif; font-size:13px;">
                    <div style="display:flex; justify-content:space-between; align-items:center;">
                        <span style="background:{badge_color}; color:#fff; padding:3px 8px; border-radius:4px; font-weight:bold;">
                            Kitchen Score: {r['final_score']}/100
                        </span>
                        <b>${r['price']}/mo</b>
                    </div>
                    <div style="margin:6px 0; font-weight:bold;">{r['address']}</div>
                    <div style="color:#444; line-height:1.5;">
                        • <b>Appliances (Stainless vs Coil):</b> {r['appliances']}/100<br>
                        • <b>Counters (Stone vs Laminate):</b> {r['counters']}/100<br>
                        • <b>Cabinets (Modern vs Dated Oak):</b> {r['cabinets']}/100<br>
                        • <b>Open Island Layout:</b> {r['layout']}/100<br>
                        • <b>Hard Flooring (No Carpet):</b> {r['flooring']}/100
                    </div>
                    <div style="margin-top:8px;">
                        <a href="{r['url']}" target="_blank" style="color:#0066cc; text-decoration:none; font-weight:bold;">Open Listing →</a>
                    </div>
                </div>
            </div>
            """
        html_doc = f"""<!DOCTYPE html>
        <html><head><meta charset="utf-8"><title>Kitchen Visual Audit Report</title></head>
        <body style="background:#f5f6f8; padding:24px; font-family:sans-serif;">
            <h2>🍳 Multi-Axis Kitchen & Interior Visual Audit Report</h2>
            <p>Showing the exact photo classified as the <b>Kitchen</b> for each candidate and its 5-axis sub-scores (weakest-link penalized):</p>
            <div style="display:flex; flex-wrap:wrap; gap:16px;">{cards_html}</div>
        </body></html>"""
        with open(export_audit_html, "w") as f:
            f.write(html_doc)
        print(f"Saved Visual Kitchen Audit card grid to {export_audit_html}!")

    df_top = df_top.copy()
    df_top.loc[df_top.head(max_candidates).index, 'vision_modern_score'] = scores
    df_top.loc[df_top.head(max_candidates).index, 'vision_notes'] = notes
    return df_top


PREFS = load_preferences(PREF_FILE)
liked_urls = PREFS['taste_profile'].get('liked_kitchen_photo_urls', [])
disliked_urls = PREFS['taste_profile'].get('disliked_kitchen_photo_urls', [])
print(f'Calibrating CLIP Vision with {len(liked_urls)} Liked and {len(disliked_urls)} Disliked kitchen photos from memory...')

scored_df = score_photos_with_local_clip(
    scored_df,
    max_candidates=25,
    liked_image_urls=liked_urls,
    disliked_image_urls=disliked_urls,
    export_audit_html='outputs/visual_kitchen_audit.html'
)

# Blend NLP text score (25%) + Strict 5-Axis Kitchen Vision score (75%)
has_v = scored_df['vision_modern_score'].notna()
scored_df.loc[has_v, 'modern_interior_score'] = (
    0.25 * scored_df.loc[has_v, 'modern_interior_score'] +
    0.75 * scored_df.loc[has_v, 'vision_modern_score']
).round(1)

weights = CONFIG['weights']
scored_df['composite_score'] = sum(scored_df.get(k, 50.0) * w for k, w in weights.items()).round(1)
scored_df = scored_df.sort_values(by='composite_score', ascending=False).reset_index(drop=True)

os.makedirs('outputs', exist_ok=True)
scored_df.to_csv('outputs/san_antonio_ranked_rentals_v4.csv', index=False)
print('Saved outputs/san_antonio_ranked_rentals_v4.csv and outputs/visual_kitchen_audit.html!')
scored_df[['composite_score', 'formatted_address', 'list_price', 'property_age_years', 'modern_interior_score', 'vision_notes', 'swri_rush_hour_mins', 'nearest_grocery']].head(15)

Calibrating CLIP Vision with 0 Liked and 0 Disliked kitchen photos from memory...
Loading Robust Multi-Axis CLIP Vision Engine on cpu...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Saved Visual Kitchen Audit card grid to outputs/visual_kitchen_audit.html!
Saved outputs/san_antonio_ranked_rentals_v4.csv and outputs/visual_kitchen_audit.html!


,composite_score,formatted_address,list_price,property_age_years,modern_interior_score,vision_notes,swri_rush_hour_mins,nearest_grocery
0,76.6,"5142 Kelwood St Unit 102, San Antonio, TX, 78250",1600,1,75.2,Kitchen 77.0/100 [App:91 | Cnt:56 | Cab:99 | O...,15.1,H-E-B
1,72.0,"9627 Summer Vail, San Antonio, TX, 78251",1595,23,60.5,Kitchen 64.0/100 [App:83 | Cnt:45 | Cab:91 | O...,15.7,Sprouts Farmers Market
2,71.9,"4121 Copano Bay, San Antonio, TX, 78229",1700,21,69.6,Kitchen 76.2/100 [App:86 | Cnt:56 | Cab:92 | O...,17.8,NOOR GROCERY
3,71.5,"11819 Zarkava Way, San Antonio, TX, 78230",2195,2,79.2,Kitchen 82.2/100 [App:92 | Cnt:65 | Cab:100 | ...,26.1,Trader Joe's
4,71.5,"7731 Bohemian Grv, San Antonio, TX, 78227",1800,2,80.1,Kitchen 83.5/100 [App:88 | Cnt:74 | Cab:99 | O...,10.1,H-E-B Plus!
5,70.7,"5015 Shara Mist, San Antonio, TX, 78251",1495,Unknown,64.0,Kitchen 68.6/100 [App:89 | Cnt:36 | Cab:82 | O...,16.3,H-E-B
6,70.6,"1022 Avocet, San Antonio, TX, 78245",1700,21,44.8,Kitchen 43.0/100 [App:91 | Cnt:24 | Cab:59 | O...,13.1,H-E-B Plus!
7,70.1,"4010 High Ridge Cir, San Antonio, TX, 78229",1950,48,64.3,Kitchen 69.1/100 [App:70 | Cnt:77 | Cab:74 | O...,14.8,Minnano Japanese Grocery
8,68.5,"9003 Hilltop Crossing Dr, San Antonio, TX, 78251",1900,21,72.1,Kitchen 79.5/100 [App:95 | Cnt:48 | Cab:97 | O...,14.1,H-E-B
9,68.4,"2031 Silver Oaks, San Antonio, TX, 78213",1900,3,68.2,Kitchen 67.6/100 [App:88 | Cnt:60 | Cab:55 | O...,24.0,La Michoacana Meat Market


In [7]:
# @title Step 6: Teach the Tool Your Taste (Record Likes / Dislikes into Persistent Memory)
# Whenever you see a listing or kitchen photo you HATE or LOVE in `scored_df` or `visual_kitchen_audit.html`,
# call `record_feedback()` below! Next time you run Step 4 & Step 5, it will automatically block rejected
# properties and use your liked/disliked kitchen photos as CLIP similarity anchors!

# Example (Uncomment and edit to log feedback):
# record_feedback(
#     address='123 Example St, San Antonio, TX',
#     liked=False,
#     kitchen_photo_url='https://ap.rdcpix.com/...jpg',  # Right-click kitchen image in visual_kitchen_audit.html -> Copy Image Address
#     reason='Dated 1990s oak cabinets and laminate countertops',
#     path=PREF_FILE
# )
print('Ready to record feedback into', PREF_FILE)

Ready to record feedback into /content/drive/MyDrive/LoveWhereYouLive_preferences.json
